# B_S3.4.1 — Metric Selection with Clean Four-Way Split

Redesign of S3.4 with a cleaner evaluation protocol:

```
null_calibration  → fit threshold for each candidate combo
signal_train      → select metric combo (power-based ranking)
null_test         → report held-out FPR
signal_test       → report held-out power
```

Key improvement over S3.4: the selection phase (signal_train) and the
evaluation phase (null_test + signal_test) use completely disjoint data.
In S3.4, validation was used for both selection and reporting.

After the final combo is chosen, the threshold can optionally be refit
using all null cases for deployment.

In [ ]:
from pathlib import Path
from itertools import combinations
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

S1_DIR  = Path('output/S1')
S3_DIR  = Path('output/S3')
OUT_DIR = Path('output/S3.4.1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED    = 20260623
ALPHA          = 0.05
SPLIT_FRACTION = 0.5
MAX_EXACT_K    = 3
MAX_EXACT_COMBOS = 75_000
MAX_SPARSE_K   = 8
BEAM_WIDTH     = 60
NEAR_FULL_TOL  = 0.02
EXCLUDE_BANDWIDTH  = True
MERGE_DCOR_DCOV    = True

## 1. Load Data (MINE-Covered Subset)

In [ ]:
df = pd.read_parquet(S3_DIR / 'permutation_all.parquet')
Z_COLS = sorted([c for c in df.columns if c.startswith('z_')])

cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
cases_null['case_id'] = cases_null['case_id'] + cases_main['case_id'].max()

meta_cols = ['case_id', 'family_id', 'family_name', 'snr', 'spread_pattern', 'x_distribution']
for col in meta_cols:
    if col not in cases_main.columns:
        cases_main[col] = np.nan
    if col not in cases_null.columns:
        cases_null[col] = np.nan

cases_all = pd.concat([cases_main[meta_cols], cases_null[meta_cols]], ignore_index=True)
df = df.merge(cases_all, on='case_id', how='left')

is_null = df['family_id'] == 'Null'
is_const = df['spread_pattern'] == 'constant'
df['category'] = 'mean+variance'
df.loc[is_null & is_const, 'category'] = 'true_null'
df.loc[~is_null & is_const, 'category'] = 'mean_only'
df.loc[is_null & ~is_const, 'category'] = 'variance_only'

mine_z = [c for c in Z_COLS if c in ('z_mic', 'z_mas', 'z_mev', 'z_mcn')]
df = df[df[mine_z].notna().all(axis=1)].reset_index(drop=True)
print(f'MINE-covered cases: {len(df):,}')
print(df['category'].value_counts().to_string())

## 2. Four-Way Split

| Split | Source | Purpose |
|-------|--------|---------|
| `null_calibration` | 50% of true_null | Fit threshold for each combo |
| `null_test` | 50% of true_null | Report held-out FPR |
| `signal_train` | 50% of signal | Select metric combo (rank by power) |
| `signal_test` | 50% of signal | Report held-out power |

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
df['split'] = ''

# Null cases → calibration / test
null_mask = df['category'] == 'true_null'
for _, group_idx in df[null_mask].groupby('x_distribution', dropna=False).groups.items():
    group_idx = np.array(list(group_idx)).copy()
    rng.shuffle(group_idx)
    n_cal = int(round(len(group_idx) * SPLIT_FRACTION))
    if len(group_idx) > 1:
        n_cal = min(max(n_cal, 1), len(group_idx) - 1)
    df.loc[group_idx[:n_cal], 'split'] = 'null_calibration'
    df.loc[group_idx[n_cal:], 'split'] = 'null_test'

# Signal cases → train / test
signal_mask = ~null_mask
for _, group_idx in df[signal_mask].groupby(['category', 'x_distribution'], dropna=False).groups.items():
    group_idx = np.array(list(group_idx)).copy()
    rng.shuffle(group_idx)
    n_train = int(round(len(group_idx) * SPLIT_FRACTION))
    if len(group_idx) > 1:
        n_train = min(max(n_train, 1), len(group_idx) - 1)
    df.loc[group_idx[:n_train], 'split'] = 'signal_train'
    df.loc[group_idx[n_train:], 'split'] = 'signal_test'

null_cal      = df[df['split'] == 'null_calibration'].copy()
null_test     = df[df['split'] == 'null_test'].copy()
signal_train  = df[df['split'] == 'signal_train'].copy()
signal_test   = df[df['split'] == 'signal_test'].copy()

print('Split sizes:')
print(f'  null_calibration: {len(null_cal):,}')
print(f'  null_test:        {len(null_test):,}')
print(f'  signal_train:     {len(signal_train):,}')
print(f'  signal_test:      {len(signal_test):,}')

print('\nSignal train by category:')
print(signal_train['category'].value_counts().to_string())
print('\nSignal test by category:')
print(signal_test['category'].value_counts().to_string())

## 3. Metric Setup and Calibration Screen

Calibration uses `null_calibration` only.

In [ ]:
def metric_display(col):
    name = col.replace('z_', '')
    return {'abs_pearson_r': '|Pearson|', 'abs_spearman_rho': '|Spearman|',
            'abs_covariance': '|Covariance|', 'mic_minus_r2': 'MIC\u2212r\u00b2'}.get(name, name)

METRIC_DISPLAY = {c: metric_display(c) for c in Z_COLS}

METRIC_GROUPS = {}
for c in Z_COLS:
    name = c.replace('z_', '')
    if name in ('pearson', 'spearman', 'abs_pearson_r', 'abs_spearman_rho', 'abs_covariance'):
        METRIC_GROUPS[c] = 'correlation'
    elif 'dcor' in name or 'dcov' in name:
        METRIC_GROUPS[c] = 'distance'
    elif 'ep_' in name or 'pf_' in name or 'seg_' in name:
        METRIC_GROUPS[c] = 'slope'
    elif 'bin_' in name or name == 'eta2':
        METRIC_GROUPS[c] = 'bin'
    elif 'dist_' in name:
        METRIC_GROUPS[c] = 'distribution'
    elif name in ('mic', 'mas', 'mev', 'mcn', 'mic_minus_r2'):
        METRIC_GROUPS[c] = 'mine'
    elif name == 'lowess_r2':
        METRIC_GROUPS[c] = 'nonlinear'
    else:
        METRIC_GROUPS[c] = 'other'

# Calibration from null_calibration
calib_rows = []
for zc in Z_COLS:
    z_null = null_cal[zc].dropna()
    pc = zc.replace('z_', 'p_')
    p_null = null_cal[pc].dropna() if pc in null_cal.columns else pd.Series(dtype=float)
    emp_fpr = float((p_null <= ALPHA).mean()) if len(p_null) else np.nan
    calib_rows.append({
        'metric': zc, 'mean_null': float(z_null.mean()) if len(z_null) else np.nan,
        'std_null': float(z_null.std()) if len(z_null) > 1 else np.nan, 'emp_fpr': emp_fpr,
    })
calib = pd.DataFrame(calib_rows)
calib['flagged'] = (
    (calib['mean_null'].abs() > 1.0) | (calib['std_null'] < 0.3) |
    (calib['std_null'] > 3.0) | (calib['emp_fpr'] > 0.15)
)

SAFE_METRICS = calib.loc[~calib['flagged'], 'metric'].tolist()
if EXCLUDE_BANDWIDTH:
    SAFE_METRICS = [m for m in SAFE_METRICS if '_bw_' not in m]
if MERGE_DCOR_DCOV and 'z_dcov' in SAFE_METRICS and 'z_dcor' in SAFE_METRICS:
    SAFE_METRICS = [m for m in SAFE_METRICS if m != 'z_dcov']

print(f'Safe metrics: {len(SAFE_METRICS)} / {len(Z_COLS)}')
if calib['flagged'].any():
    print('Flagged:', calib.loc[calib['flagged'], 'metric'].tolist())

## 4. Evaluation Helpers

Two distinct evaluation modes:
- **Selection**: threshold from `null_cal`, power from `signal_train`
- **Held-out test**: threshold from `null_cal`, FPR from `null_test`, power from `signal_test`

In [ ]:
def fit_threshold(null_data, metrics, alpha=ALPHA):
    T_null = null_data[metrics].max(axis=1).dropna()
    if len(T_null) == 0:
        return np.nan
    return float(np.nanpercentile(T_null, 100 * (1 - alpha)))


def compute_rates(data, metrics, threshold):
    T = data[metrics].max(axis=1)
    detected = (T > threshold).fillna(False)
    out = {}
    for cat in ['true_null', 'mean_only', 'variance_only', 'mean+variance']:
        mask = data['category'] == cat
        out[f'rate_{cat}'] = float(detected[mask].mean()) if mask.sum() else np.nan
        out[f'n_{cat}'] = int(mask.sum())
    signal = data['category'] != 'true_null'
    out['overall_power'] = float(detected[signal].mean()) if signal.sum() else np.nan
    mo, vo, mv = out['rate_mean_only'], out['rate_variance_only'], out['rate_mean+variance']
    rates = [r for r in (mo, vo, mv) if r is not None and not np.isnan(r)]
    out['macro_power'] = float(np.mean(rates)) if rates else np.nan
    return out


def select_evaluate(metrics, label=None, kind=None):
    """Selection phase: threshold from null_cal, power from signal_train."""
    metrics = list(metrics)
    threshold = fit_threshold(null_cal, metrics)
    sel = compute_rates(signal_train, metrics, threshold)
    return {
        'label': label or ', '.join(METRIC_DISPLAY.get(m, m) for m in metrics),
        'kind': kind or 'combo',
        'k': len(metrics),
        'metrics': ','.join(metrics),
        'display_metrics': ', '.join(METRIC_DISPLAY.get(m, m) for m in metrics),
        'threshold': threshold,
        'select_power': sel['overall_power'],
        'select_macro': sel['macro_power'],
        'select_MO': sel['rate_mean_only'],
        'select_VO': sel['rate_variance_only'],
        'select_MV': sel['rate_mean+variance'],
    }


def holdout_evaluate(metrics, threshold=None):
    """Held-out phase: FPR from null_test, power from signal_test."""
    metrics = list(metrics)
    if threshold is None:
        threshold = fit_threshold(null_cal, metrics)
    null_res = compute_rates(null_test, metrics, threshold)
    sig_res = compute_rates(signal_test, metrics, threshold)
    return {
        'threshold': threshold,
        'test_FPR': null_res['rate_true_null'],
        'test_power': sig_res['overall_power'],
        'test_macro': sig_res['macro_power'],
        'test_MO': sig_res['rate_mean_only'],
        'test_VO': sig_res['rate_variance_only'],
        'test_MV': sig_res['rate_mean+variance'],
    }


def print_select_table(table, max_rows=30):
    cols = ['label', 'k', 'select_power', 'select_macro',
            'select_MO', 'select_VO', 'select_MV']
    available = [c for c in cols if c in table.columns]
    print(table[available].head(max_rows).to_string(
        index=False, float_format='{:.3f}'.format))

## 5. Full-Combo Reference (on signal_train)

In [ ]:
ref_all = select_evaluate(Z_COLS, label='all_metrics', kind='reference')
ref_safe = select_evaluate(SAFE_METRICS, label='all_safe', kind='reference')
references = pd.DataFrame([ref_all, ref_safe])
print_select_table(references)

FULL_SAFE_POWER = ref_safe['select_power']
print(f'\nFull-combo reference power (signal_train): {FULL_SAFE_POWER:.3f}')

## 6. Individual Metrics (k=1)

Ranked by signal_train power.

In [ ]:
individual_rows = []
for metric in SAFE_METRICS:
    row = select_evaluate([metric], label=METRIC_DISPLAY[metric], kind='individual')
    row['group'] = METRIC_GROUPS[metric]
    individual_rows.append(row)

individual = pd.DataFrame(individual_rows).sort_values('select_power', ascending=False)
individual.to_csv(OUT_DIR / 'individual_metrics.csv', index=False, float_format='%.6f')

print('Top individual metrics (signal_train power):')
print(individual[['label', 'group', 'k', 'select_power', 'select_macro',
                  'select_MO', 'select_VO', 'select_MV']]
      .head(15).to_string(index=False, float_format='{:.3f}'.format))

## 7. Exact Best Subset (k=1, 2, 3)

In [ ]:
exact_rows = []
for k in range(1, min(MAX_EXACT_K, len(SAFE_METRICS)) + 1):
    n_combos = int(math.comb(len(SAFE_METRICS), k))
    if n_combos > MAX_EXACT_COMBOS:
        print(f'Skip exact k={k}: {n_combos:,} > {MAX_EXACT_COMBOS:,}')
        continue

    print(f'Exact k={k}: testing {n_combos:,} combinations')
    best_row = None
    best_power = -np.inf

    for combo in combinations(SAFE_METRICS, k):
        row = select_evaluate(combo, kind='exact')
        if row['select_power'] > best_power:
            best_power = row['select_power']
            best_row = row

    best_row['label'] = f'exact best k={k}'
    exact_rows.append(best_row)
    print(f'  Best: {best_row["display_metrics"]}')
    print(f'  signal_train power={best_row["select_power"]:.3f}')

exact_best = pd.DataFrame(exact_rows)
exact_best.to_csv(OUT_DIR / 'exact_best_subset.csv', index=False, float_format='%.6f')
print('\nExact best:')
print_select_table(exact_best)

## 8. Backward Elimination

All decisions based on signal_train power.

In [ ]:
remaining = list(SAFE_METRICS)
backward_rows = []

row0 = select_evaluate(remaining, label=f'backward k={len(remaining)} (full)', kind='backward')
row0['removed_display'] = '(start)'
backward_rows.append(row0)

while len(remaining) > 1:
    best_row = None
    best_remove = None
    best_power = -np.inf

    for candidate in remaining:
        combo = [m for m in remaining if m != candidate]
        row = select_evaluate(combo, kind='backward')
        if row['select_power'] > best_power:
            best_power = row['select_power']
            best_row = row
            best_remove = candidate

    remaining.remove(best_remove)
    k = len(remaining)
    best_row['label'] = f'backward k={k}'
    best_row['removed_display'] = METRIC_DISPLAY[best_remove]
    backward_rows.append(best_row)

    if k <= 10 or k % 5 == 0:
        print(f'k={k:2d}: remove {METRIC_DISPLAY[best_remove]:25s} -> power={best_row["select_power"]:.4f}')

backward_path = pd.DataFrame(backward_rows)
backward_path.to_csv(OUT_DIR / 'backward_elimination.csv', index=False, float_format='%.6f')

peak_idx = backward_path['select_power'].idxmax()
peak = backward_path.loc[peak_idx]
print(f'\nPeak signal_train power at k={int(peak["k"])}: {peak["select_power"]:.4f}')
print(f'  metrics: {peak["display_metrics"]}')

## 9. Beam Search

In [ ]:
beam_rows = []
beam = [tuple()]

for k in range(1, min(MAX_SPARSE_K, len(SAFE_METRICS)) + 1):
    candidates = set()
    for combo in beam:
        for metric in SAFE_METRICS:
            if metric not in combo:
                candidates.add(tuple(sorted(combo + (metric,))))

    print(f'Beam k={k}: evaluating {len(candidates):,} candidates')
    scored = [select_evaluate(combo, kind='beam') for combo in candidates]
    scored_df = pd.DataFrame(scored).sort_values('select_power', ascending=False)
    beam = [tuple(row.split(',')) for row in scored_df['metrics'].head(BEAM_WIDTH)]

    best_row = scored_df.iloc[0].to_dict()
    best_row['label'] = f'beam best k={k}'
    beam_rows.append(best_row)
    print(f'  Best: {best_row["display_metrics"]}  power={best_row["select_power"]:.3f}')

beam_best = pd.DataFrame(beam_rows)
beam_best.to_csv(OUT_DIR / 'beam_search.csv', index=False, float_format='%.6f')
print('\nBeam best:')
print_select_table(beam_best)

## 10. Recommendation (from signal_train)

Selection rule:
1. Prefer combos with signal_train power within `NEAR_FULL_TOL` of full-combo reference.
2. Among those, choose smallest k.
3. Tie-breaker: higher power.

In [ ]:
all_sparse = pd.concat([exact_best, backward_path, beam_best], ignore_index=True, sort=False)
all_sparse = all_sparse.drop_duplicates(subset=['metrics'], keep='first')
all_sparse['gap_vs_full'] = FULL_SAFE_POWER - all_sparse['select_power']
all_sparse.to_csv(OUT_DIR / 'all_sparse_candidates.csv', index=False, float_format='%.6f')

near_full = all_sparse[all_sparse['gap_vs_full'] <= NEAR_FULL_TOL].copy()
if near_full.empty:
    best_val = all_sparse['select_power'].max()
    near_full = all_sparse[all_sparse['select_power'] >= best_val - NEAR_FULL_TOL].copy()
    print('No combo within tolerance of full-combo; selecting near best.')

recommended = near_full.sort_values(
    ['k', 'select_power'], ascending=[True, False]
).iloc[0]
recommended_metrics = recommended['metrics'].split(',')

print('Recommended combo (selected on signal_train):')
print(f'  k = {int(recommended["k"])}')
print(f'  metrics = {recommended["display_metrics"]}')
print(f'  signal_train power = {recommended["select_power"]:.3f}')
print(f'  signal_train macro = {recommended["select_macro"]:.3f}')

## 11. Held-Out Evaluation (null_test + signal_test)

The recommended combo and key alternatives are evaluated on completely
held-out data. This is the unbiased estimate of real-world performance.

In [ ]:
# Collect combos to evaluate on held-out data
test_combos = {}
test_combos[f'recommended (k={len(recommended_metrics)})'] = recommended_metrics

for _, row in exact_best.iterrows():
    k = int(row['k'])
    test_combos[f'exact best k={k}'] = row['metrics'].split(',')

for k_target in [2, 3, 5, 10]:
    bk = backward_path[backward_path['k'] == k_target]
    if len(bk):
        test_combos[f'backward k={k_target}'] = bk.iloc[0]['metrics'].split(',')

test_combos['all_safe'] = SAFE_METRICS

# Evaluate each on held-out splits
test_rows = []
for label, metrics in test_combos.items():
    ho = holdout_evaluate(metrics)
    test_rows.append({
        'label': label,
        'k': len(metrics),
        'threshold': ho['threshold'],
        'test_FPR': ho['test_FPR'],
        'test_power': ho['test_power'],
        'test_macro': ho['test_macro'],
        'test_MO': ho['test_MO'],
        'test_VO': ho['test_VO'],
        'test_MV': ho['test_MV'],
    })

test_table = pd.DataFrame(test_rows)
test_table.to_csv(OUT_DIR / 'holdout_evaluation.csv', index=False, float_format='%.6f')

print('Held-out evaluation (null_test FPR + signal_test power):')
print(test_table.to_string(index=False, float_format='{:.3f}'.format))

## 12. Refit Threshold with All Null (for Deployment)

After selecting the final combo, use ALL null cases for the most stable threshold estimate.

In [ ]:
all_null = df[df['category'] == 'true_null']
threshold_cal = fit_threshold(null_cal, recommended_metrics)
threshold_final = fit_threshold(all_null, recommended_metrics)

print(f'Recommended combo: {recommended["display_metrics"]}')
print(f'  Threshold (null_calibration, n={len(null_cal):,}): {threshold_cal:.4f}')
print(f'  Threshold (all null,         n={len(all_null):,}): {threshold_final:.4f}')
print(f'  Difference: {abs(threshold_final - threshold_cal):.4f}')

with open(OUT_DIR / 'recommendation.txt', 'w') as f:
    f.write('B_S3.4.1 Metric Selection Recommendation\n')
    f.write(f'metrics = {recommended["display_metrics"]}\n')
    f.write(f'k = {int(recommended["k"])}\n')
    f.write(f'threshold_calibration = {threshold_cal:.6f}\n')
    f.write(f'threshold_final_all_null = {threshold_final:.6f}\n')
    rec_test = test_rows[0]
    f.write(f'test_FPR = {rec_test["test_FPR"]:.6f}\n')
    f.write(f'test_power = {rec_test["test_power"]:.6f}\n')
    f.write(f'test_macro = {rec_test["test_macro"]:.6f}\n')
    f.write(f'test_MO = {rec_test["test_MO"]:.6f}\n')
    f.write(f'test_VO = {rec_test["test_VO"]:.6f}\n')
    f.write(f'test_MV = {rec_test["test_MV"]:.6f}\n')
print(f'\nSaved to {OUT_DIR / "recommendation.txt"}')

## 13. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Backward elimination path
ax = axes[0]
ax.plot(backward_path['k'], backward_path['select_power'], '-o',
        color='#377eb8', markersize=4, label='backward (signal_train)')
ax.axhline(FULL_SAFE_POWER, color='black', ls='--', alpha=0.5, label='full-combo reference')
ax.set_xlabel('Number of metrics remaining')
ax.set_ylabel('Power (signal_train)')
ax.set_title('Backward Elimination')
ax.set_ylim(0.85, 1.005)
ax.legend(fontsize=8)
ax.invert_xaxis()

# Zoom k=1..15
ax = axes[1]
zoom = backward_path[backward_path['k'] <= 15]
ax.plot(zoom['k'], zoom['select_power'], '-o', color='#377eb8', markersize=5, label='backward')
ax.plot(exact_best['k'], exact_best['select_power'], '^', color='#333333', markersize=8, label='exact best')
beam_zoom = beam_best[beam_best['k'] <= 15]
ax.plot(beam_zoom['k'], beam_zoom['select_power'], 'D', color='#4daf4a', markersize=6, label='beam best')
ax.axhline(FULL_SAFE_POWER, color='black', ls='--', alpha=0.5, label='full-combo reference')
ax.set_xlabel('Number of metrics remaining')
ax.set_ylabel('Power (signal_train)')
ax.set_title('Zoom: k=1..15')
ax.set_ylim(0.93, 1.002)
ax.set_xticks(range(1, 16))
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(OUT_DIR / 'fig1_backward_elimination.png', dpi=150, bbox_inches='tight')
plt.show()

# Held-out comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(test_table))
width = 0.35
ax.bar([i - width/2 for i in x], test_table['test_power'], width,
       label='test_power', color='#377eb8', alpha=0.8)
ax.bar([i + width/2 for i in x], test_table['test_FPR'], width,
       label='test_FPR', color='#e41a1c', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(test_table['label'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Rate')
ax.set_title('Held-Out Evaluation: Power and FPR by Combo')
ax.axhline(ALPHA, color='gray', ls=':', alpha=0.5)
ax.legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig2_holdout_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Summary

In [ ]:
print('=' * 80)
print('B_S3.4.1 Metric Selection Summary (Clean Four-Way Split)')
print('=' * 80)
print(f'Cases: {len(df):,} (MINE-covered subset)')
print(f'  null_calibration: {len(null_cal):,} (fit threshold)')
print(f'  null_test:        {len(null_test):,} (report FPR)')
print(f'  signal_train:     {len(signal_train):,} (select combo)')
print(f'  signal_test:      {len(signal_test):,} (report power)')
print(f'Safe metrics: {len(SAFE_METRICS)}')
print()

print('── Selection (signal_train) ──')
max_k = int(backward_path['k'].max())
bk_key = backward_path[backward_path['k'].isin([max_k, 15, 10, 5, 3, 2, 1])]
print(bk_key[['label', 'k', 'select_power', 'select_macro']]
      .to_string(index=False, float_format='{:.4f}'.format))
print()

print('── Held-out evaluation (null_test + signal_test) ──')
print(test_table.to_string(index=False, float_format='{:.4f}'.format))
print()

rec_test = test_rows[0]
print('── Recommendation ──')
print(f'  metrics:    {recommended["display_metrics"]}')
print(f'  k:          {int(recommended["k"])}')
print(f'  test FPR:   {rec_test["test_FPR"]:.3f}')
print(f'  test power: {rec_test["test_power"]:.3f}')
print(f'  test macro: {rec_test["test_macro"]:.3f}')
print(f'  test MO/VO/MV: {rec_test["test_MO"]:.3f} / {rec_test["test_VO"]:.3f} / {rec_test["test_MV"]:.3f}')
print(f'  threshold (null_cal):  {threshold_cal:.4f}')
print(f'  threshold (all null):  {threshold_final:.4f}')
print(f'\nSaved to {OUT_DIR}')